# Sliding Window
- for characters problems

In [2]:
from typing import List, Optional

<b> 3. Longest Substring without repeating characters </b>  
keep a hash map to track all visited characters, the search in a hashmap is O(1). char as key, index as value.  
two pointers to construct a <b> sliding window </b>, i is the start of the substring, j keeps moving forward. if j meets a used char in hashmap, i starts to moving forward and updating the hashmap by removing the chars until deleting the duplicated char that j now points. now i is at right of the last found duplicated char.  
the char between i and j have already checked no duplicates, so j just keep moving forward, no need to re-check chars between new i and j.

The two pointers only move forward
O(2n) = 2(N)

In [1]:
def lengthOfLongestSubstring(s):
    n = len(s)
    i, j = 0, 0
    result = 0
    temp = {}   # visisted chars, char as key, index as value
    while j < n:
        if s[j] not in temp:
            temp[s[j]] = j
            j+=1
            result = max(result, j-i)
        else:
            del temp[s[i]]
            i += 1
    return result

In [2]:
# this solution further optimize the previous one, by locating the i quickly.

def lengthOfLongestSubstring(s: str) -> int:
    result = 0
    visited = {} # char: latest_index
    i = 0 # start of substring
    for j, char in enumerate(s):
        if char in visited and i <=visited[char]:
            # duplicate char appear after the start of substring => need to update the start i
            i = visited[char]+1
        visited[char] = j
        result = max(result, j-i+1)
    return result
    

In [3]:
s="abcabcbb"
lengthOfLongestSubstring(s)

3

<b> 159 Longest substring with at most 2 distinct characters </b>

also sliding window, or two pointers.

using a dictionary to keep track of the {char: last_ind}. when the distinct char size is > 2, find the first distinct char position, remove that char from the dictionary, and update the left pointer to the position, similar to the problem 3 of how the <b>i</b> is updated quickly. because only 3 item in the dictionary, a min() search is simiar to O(1)

time complexity:
- O(N) for looping over the s
- O(1) for min(3) and del item from dictionary
- so in total O(N)

In [4]:

def lengthOfLongestSubstringTwoDistinct(s: str) -> int:
    left = 0
    c_dict = {}
    max_l = 0
    for i, c in enumerate(s):
        c_dict[c] = i
        # i keeps moving until hit the third distinct char
        if len(c_dict) == 3:
            # remove the first distinct char and move the left pointer forward
            del_ind = min(c_dict.values())
            # s[del_ind] locate the char, which is the key in c_dict
            c_dict.pop(s[del_ind])
            left = del_ind+1
        max_l = max(max_l, i-left+1)
    return max_l

<b> 340. longest substring with at most K distinct characters </b> 

step 1, same as problem 159, but using a while loop to reduce the size of dictionary until it is within limit.

time complexity:
-   O(N) for loop
-   within the loop, the while loop actually only takes O(1) since K is a constant then min() takes constant O(K) which is O(1). worse case for every c from words, the while loop is triggered. so worse case thie while loop in total takes O(N) for entire outer loop
-   in total O(N)+O(N) = O(N)

space complexity:
-   O(K) for the dictionary

In [5]:
from collections import defaultdict

def lengthOfLongestSubstringKDistinct(s: str, k: int) -> int:
    visited = defaultdict(int) # {char: last_index}
    i = 0
    res = 0
    for j, c in enumerate(s):
        visited[c] = j
        # update i
        while len(visited)>k:
            del_ind = min(visited.values())
            del visited[s[del_ind]]
            i = del_ind +1
        res = max(res, j-i+1)
    return res     

below is the initial solution, that is not optimized yet

In [7]:
from collections import defaultdict

def lengthOfLongestSubstringKDistinct(s: str, k: int) -> int:
    """
    also O(N) time complexity but slower than previous solution
    the total number of steps the inner while loop performs across all iterations of the outer loop is O(n)
    because i only move forward once, at most move N times
    """
    visited = defaultdict(int) # {char: count of occurance}
    i = 0
    res = 0 

    for j, c in enumerate(s):
        if visited[c] == 0: k-=1
        visited[c] += 1
        while k < 0:
            # update i to be valid again
            visited[s[i]] -= 1
            if visited[s[i]] == 0: k += 1
            i += 1
        
        res = max(res, j-i+1)
    return res
        
        

<b> 992 subarrays with K different integers </b>

write a helper function to return number of array with at most K different characters

In [9]:
class Solution:
    def subarraysWithKDistinct(self, nums: List[int], k: int) -> int:
        def atMostK(nums, k):
            """
            return the count of valid substring with at most k distinct characters
            """
            i = 0 # starting index
            res = 0 
            count_map = defaultdict(int)
            for j, c in enumerate(nums):
                if count_map[c] == 0: k-=1
                count_map[c]+=1
                while k < 0:
                    # find the valid i
                    # can optimize by saving seen dictionary as {char: last_index} then using min
                    count_map[nums[i]]-=1
                    if count_map[nums[i]] == 0: k+=1
                    i += 1
                # now i is a valid staring position of subarray
                # j-i+1 is the new count of valid subarrays with ind j as the ending position.
                # [i, i+1, i+2, j] the new subarray ending at j starting at i is [j], [i+2, j], [i+1 to j], [i to j] 
                # exactly the size of the j-i+1 
                # this satisfy the purpose of finding subarray with AT MOST K distinct characters
                # because from [i to j] at most there are k distinct chars, any subarray within it will satisfy that constraint too, maybe less than K
                # this can avoid duplicated count of subarray ending at prev j position
                res += (j-i+1)
            return res
        # at most k - at most k-1 becomes exact k
        return atMostK(nums, k) - atMostK(nums, k-1)
        

<b> 2260 minimum consecutive cards to pick up </b>

find the minimum distance between two cards with same value. using a dictionary saving the index of each char when looping over the whole string can solve that. Time complexity O(N)

In [10]:
import math
def minimumCardPickup(cards: List[int]) -> int:
    """
    sliding window
    """

    ind_map = {} # char: most recent_ind
    res = math.inf
    for i, c in enumerate(cards):
        if c in ind_map:
            res = min(res, i - ind_map[c]+1)
        ind_map[c] = i
    return res if res < math.inf else -1

<b> 2401 longest nice subarray </b>

bit operation of two integers
-   &: compare each bit of two numbers and returns 1 if both corresponding bits are 1 otherwise it returns 0
    -   6: 110, 3: 011 => 6&3 = 010 -> 2
-   |: compare each bit of two numbers and return 1 if at least one of the bit is 1, otherwise 0
    -   6: 110, 3: 011  -> 6|3 = 111 -> 7 = (1*2**2 + 1*2**1 + 1*2**0)
-   ^: XOR, compare each bit of 2 numbers and return 1 if the bit are different at the same position, otherwise 0
    -   6: 110, 3: 011  -> 6^3=101 -> 1*2**2+0*2**1+1*2**0 = 5
    -   same as doing extraction



In [11]:

def longestNiceSubarray(nums: List[int]) -> int:
    # AND: check all values in the current window
    # i: left windown position
    res = AND = i = 0
    for j in range(len(nums)):
        # when AND & nums[j] == 0, 
        # j can be in the nice subarray
        # otherwise window shrink by ++i, and the ith element
        # need to be removed out from AND
        while AND & nums[j]:
            # ^ operation can remove the item by offsetting
            AND = AND ^ nums[i]
            i += 1
        # add item nums[j] to the AND var by | operation
        AND = AND | nums[j]
        res = max(res, j-i+1)
    return res


<b> 2981 find longest special substring that occurs trice I </b>

save {(char, count): freq }.  
to ensure all occurance is captured, using another pointer j to move forward .

time complexity:
-   outer loop O(N)
-   innner loop at most O(N) at worse case
-   here the for loop in total is O(N**2)
-   for the map, its size at worse case is proportional to N
-   so total O(N ** 2)+O(N) = O(N ** 2)

time complexity, why in worse case the hash_map takes O(N)
-  for example, if all char in string are distinct: "abcde", key number is 5, N
-  if all char in string are the same "aaaaa", distinct keys are: a, aa, aaa, aaaa, aaaaa, also capped by size of the string N

In [12]:
from collections import Counter, defaultdict

def maximumLength(s: str) -> int:
    hash_map = defaultdict(int)
    n = len(s)
    for i in range(n):
        count = 1
        hash_map[(s[i], count)] += 1
        j = i
        while j+1 < n:
            # j initilize at i, so try to find how many concecutive s[i] here is
            if s[j] == s[j+1]:
                count += 1
                hash_map[(s[i], count)] += 1
                j+=1
            else:
                break
    res = -1
    for k, v in hash_map.items():
        if v >=3:
            res = max(res, k[1])
    return res


    
    

<b> 2982 find longest special substring that occurs trice II </b>

the only difference from the above question is the size of the array is very large and need to optmize the solution to be much more efficient since the O(N^2) solution above will cause time exceeds error

here the hash_map only store the accurance when the distinct char*count appeared.

for the actual freq of all the valid subsequence it is in the second for loop of the hash_map. The accumulative count from longest subsequence to shorter one is smart to keep track of the freq.
-   hash_map["a"][3] = 1, "aaa" but we know that if ending at third a, there is also valid substring of "a" (s[2]), "aa"(s[1:3]), and "aaa"(s[0:3]) all ending at pos 2.
-   same for hash_map["a"][2]=1, "aa", we know if ending at second a, there is also valid substring "a": (s[1]), then "aa": s[0:2] ending at pos 2. 
-   in this way when looping from longer substring to shorter substring, the accu sum can help to get the actual freq account, like "aa" is from map["a"][2] and map["a"][3]
-   the break is smart to stop the checking earilier
-   also the starting range of count is from max(hash_map[char].keys())


time complexity:
-   first loop to populate the hashmap takes O(N)
-   second for loop of the hashmap
    -   if all char are distinct, the size of hash_map is N, but the count loop will be O(1)
    -   if all chars are the same, the size of hash_map is 1, but the count loop will be O(N), it can end early because of the "break" sentence
    -   so roughly the second for loop, even though a nested one, still takes O(N)
-   so  total O(N)

space complexity of O(N), the items of the hash_map is proportional to N


In [ ]:
class Solution:
    def maximumLength(self, s: str) -> int:
        hash_map = defaultdict(lambda: defaultdict(int)) # {char:{len: freq}}
        n = len(s)
        count = 0
        for i in range(n):
            if i == 0 or s[i]!=s[i-1]:
                count = 1
            else:
                count += 1
                # only save the distinct count pair encountered
            hash_map[s[i]][count]+=1

        res = -1
        for char in hash_map:
            acc_sum = 0
            for count in range(max(hash_map[char].keys()), 0, -1):
                # count is capped by size n
                # get freq of (char*count) 
                # if aaa, first the freq of "aaa" is 1, 
                # then "aa" actually can be from s[0:2] -> ending at second a, and s[1:3] -> ending at third a. the count of string ending at third a is from hash_map[a][3]
                acc_sum += hash_map[char][count]
                if acc_sum >=3:
                    res = max(res, count)
                    # the smaller count value also has acc_sum >=3
                    # but no need to check them
                    # since it wont change the res value
                    break
        return res


<b> 485 Max consecutive ones </b>

option 1, reset starting pointer i whenever met a 0

In [ ]:
def findMaxConsecutiveOnes(nums: List[int]) -> int:
    res = 0
    i = -1
    for j in range(len(nums)):
        if nums[j] == 0:
            # reset i
            i = j
        else:
            res = max(res, j-i)
    return res

In [13]:
# another way that can be used in follow up problems:

def findMaxConsecutiveOnes(nums: List[int]) -> int:
    i = 0
    zero_count = 0 # no zero allowed.
    for j in range(len(nums)):
        if nums[j] == 0:
            zero_count += 1
        if zero_count > 0:
            zero_count -= (nums[i]==0) 
            i += 1

    return j - i+1

<b> 487 max consecutive ones II </b>

using the second solution but now change the allowed zero value to 1

In [14]:

def findMaxConsecutiveOnes(self, nums: List[int]) -> int:
    """
    using sliding window, 

    follow up quesiton of nums being an infinite stream: store the index of value 0
    """
    i = 0
    flag = 1
    for j in range(len(nums)):
        if nums[j] == 0:
            # flip the value
            flag -= 1
        if flag < 0:
            flag += 1- nums[i]
            i += 1
    return j - i + 1
        

<b> 1004 Max consecutive ones III </b>

now the allowed 0 value is K, same solution as above

In [15]:

def longestOnes(nums: List[int], k: int) -> int:
    """
    the problem is transformed into:
    find the longest subarray with at most K 0s
    using sliding window: 
    - if num_of_zero <= K, increment j
    - if num_of_zero > K, increment i and j
    """
    i = 0
    # no need to have another res var to keep track of the longest subarray length, the i and j sliding with the limit of k will find its longest subarray at the end
    for j in range(len(nums)):
        # one zero is found
        if nums[j] == 0: 
            k-=1
        # no quota left, moving i potentially increase k
        if k < 0:
            k += 1-nums[i] # k may or may not increase
            i+=1
    return j - i+1


<b> 128 longest consecutive sequence </b>

besides the solution using hashmap (recorded in the HashTable notebook), another way is directly looping over the number, and only once. (because of checking x-1)

In [4]:
# this one is not easy to understand and explain


def longestConsecutive(nums: List[int]) -> int:
    """
    using a hashmap to store {num: count of consecutive neighbors so far}
    remember to upate the left most and right most boundry's count, 
    so later on new element can be concatenated without extra pain 
    """
    hash_map = {} # num: count of consecutive neighbor
    res = 0
    for n in nums:
        if n not in hash_map:
            left = hash_map.get(n-1, 0)
            right = hash_map.get(n+1, 0)
            hash_map[n] = left+right+1
            
            res = max(res, hash_map[n])
            
            # update the boundry too
            # left: how many left consecutive num
            # right: how many right consecutive num
            # n - left: the starting number of the sequence
            # n + right: the ending number of the sequence
            hash_map[n-left] = hash_map[n]
            hash_map[n+right] = hash_map[n]
    return res

In [5]:
# this solution is better
class Solution:
    def longestConsecutive(self, nums: List[int]) -> int:
        """
        this solution is still O(N) even though the while loop, 
        because the while loop only starts for x-1 not in nums
        """
        nums = set(nums)
        best = 0
        for x in nums:
            if x-1 not in nums:
                # meaning x is the starting point of the sequence
                y = x+1
                # check where y can reach
                while y in nums:
                    y+=1
                best = max(best, y-x)
        return best